In [3]:
import rasterio
import numpy as np

In [4]:
with rasterio.open("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/vs30/vs30_mosaic.tif") as src:
    print(f"CRS:        {src.crs}")
    print(f"Resolution: {src.res}")
    print(f"Dimensions: {src.width} x {src.height}")
    print(f"Bounds:     {src.bounds}")
    print(f"Bands:      {src.count}")
    print(f"Dtype:      {src.dtypes}")
    print(f"Nodata:     {src.nodata}")
    
    # Read a sample to check value ranges
    data = src.read(1)
    valid = data[data != src.nodata] if src.nodata else data[data > 0]
    print(f"\nValue range (valid pixels):")
    print(f"  Min:  {valid.min():.1f} m/s")
    print(f"  Max:  {valid.max():.1f} m/s")
    print(f"  Mean: {valid.mean():.1f} m/s")
    print(f"  % nodata: {100*(1 - len(valid)/data.size):.1f}%")

CRS:        EPSG:4326
Resolution: (0.00833333333333, 0.00833333333333)
Dimensions: 43201 x 16801
Bounds:     BoundingBox(left=-180.00416666666666, bottom=-56.00416666666666, right=180.00416666652268, top=84.00416666661067)
Bands:      1
Dtype:      ('int32',)
Nodata:     None

Value range (valid pixels):
  Min:  1.0 m/s
  Max:  2197.0 m/s
  Mean: 570.0 m/s
  % nodata: 0.0%


In [6]:
with rasterio.open("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/vs30/vs30_mosaic.tif") as src:
    data = src.read(1)
    
    # Check suspicious low values
    print("Value distribution at low end:")
    for threshold in [0, 1, 2, 5, 10, 50, 100]:
        count = (data <= threshold).sum()
        print(f"  <= {threshold:>6}: {count:>10} pixels ({100*count/data.size:.2f}%)")
    
    print("\nValue distribution at high end:")
    for threshold in [1500, 1800, 2000, 2100, 2197]:
        count = (data >= threshold).sum()
        print(f"  >= {threshold:>6}: {count:>10} pixels ({100*count/data.size:.2f}%)")
    
    # Check what values appear in a known ocean region
    # Pacific Ocean around 0N, 180W — should be nodata
    with rasterio.open("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/vs30/vs30_mosaic.tif") as src:
        from rasterio.windows import from_bounds
        ocean_window = from_bounds(-170, -10, -160, 0, src.transform)
        ocean_sample = src.read(1, window=ocean_window)
        print(f"\nOcean sample (Pacific, should be nodata):")
        print(f"  Unique values: {np.unique(ocean_sample)}")
        print(f"  Min: {ocean_sample.min()}, Max: {ocean_sample.max()}")

Value distribution at low end:
  <=      0:         25 pixels (0.00%)
  <=      1:         79 pixels (0.00%)
  <=      2:        110 pixels (0.00%)
  <=      5:        192 pixels (0.00%)
  <=     10:        436 pixels (0.00%)
  <=     50:       2022 pixels (0.00%)
  <=    100:      12371 pixels (0.00%)

Value distribution at high end:
  >=   1500:      46126 pixels (0.01%)
  >=   1800:      33259 pixels (0.00%)
  >=   2000:      25918 pixels (0.00%)
  >=   2100:      22009 pixels (0.00%)
  >=   2197:      16750 pixels (0.00%)

Ocean sample (Pacific, should be nodata):
  Unique values: [180 199 219 228 600]
  Min: 180, Max: 600


In [7]:
patches = {
    "Kanto_Japan":   (138.5, 34.5, 141.5, 37.2),   # expect ~150-400 (sedimentary basin)
    "W_Australia":   (117.0, -32.0, 120.5, -29.0),  # expect ~600-800 (hard craton)
    "Central_Chile": (-72.5, -36.5, -69.5, -33.5),  # expect ~300-600 (mixed)
    "Ordos_China":   (107.5, 37.0, 111.0, 40.0),    # expect ~400-600 (loess plateau)
}

with rasterio.open("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/vs30/vs30_mosaic.tif") as src:
    for name, (left, bottom, right, top) in patches.items():
        window = from_bounds(left, bottom, right, top, src.transform)
        data = src.read(1, window=window)
        print(f"{name:<25} min={data.min():>5}  max={data.max():>5}  mean={data.mean():>6.1f} m/s")

Kanto_Japan               min=  126  max=  900  mean= 532.5 m/s
W_Australia               min=  180  max= 1100  mean= 441.4 m/s
Central_Chile             min=  180  max=  900  mean= 622.3 m/s
Ordos_China               min=  180  max=  852  mean= 354.4 m/s


## Insights

<p>
The USGS Global Vs30 mosaic covers the full global extent (180°W–180°E, 56°S–84°N) at approximately 30 arcsec (~900m) resolution in WGS84/EPSG:4326, making it directly compatible with all other layers without reprojection. The dataset is stored as int32 with no nodata gaps. It is a genuinely global mosaic that assigns Vs30 estimates everywhere including ocean floor, using bathymetric slope as a proxy for submarine sediment conditions. This eliminates any nodata masking requirement during processing and simplifies the rasterisation pipeline considerably.
</p>

<p>
Global value range is 1–2197 m/s with a mean of 570 m/s. Anomalously low values (≤100 m/s) account for just 0.00% of pixels and represent genuinely soft or water-saturated sediments rather than data errors. The upper tail (≥2197 m/s, ~16,750 pixels) corresponds to exposed hard bedrock outcrops and is physically legitimate. The ocean sample check confirmed that the mosaic assigns real sediment-proxy values (180–600 m/s) to oceanic regions rather than leaving them blank. They are consistent with the USGS hybrid mosaic design.
</p>

<p>
Patch-level inspection reveals physically interpretable spatial patterns that are encouraging for the geological clustering step. Ordos China records the lowest mean (354 m/s), consistent with thick Loess Plateau sediment cover dampening shear wave velocities. Central Chile returns the highest mean (622 m/s), reflecting the dominance of hard Andean bedrock in the eastern portion of the patch. Kanto Japan shows a wide range (126–900 m/s) with a mean of 532 m/s. It higher than expected given the soft Tokyo basin sediments, suggesting the patch mean is being pulled upward by the hard Kanto mountain terrain to the west. The within-patch variance for Kanto will be high, which is useful for clustering but should be noted when interpreting cell-level predictions in the basin versus the surrounding uplands.
</p>

<p>
Western Australia is the one patch warranting a flag. The mean of 441 m/s is lower than the 600–800 m/s typically associated with exposed Archean craton, and the 180 m/s minimum is anomalous for a stable cratonic setting. The most likely explanation is that the USGS slope-based Vs30 proxy underestimates values on very flat cratonic terrain where topographic gradients are near zero, or that coastal sediments on the western edge of the patch are pulling the mean down. This will be monitored during the clustering step, if Western Australia cells cluster with soft-sediment patches rather than stable craton patches, a correction or manual override of the Vs30 values for this patch may be warranted.
</p>

<p>
Overall the Vs30 layer is clean, globally complete, and ready for processing. The per-patch value distributions are physically interpretable and the spatial patterns align with expected geological conditions across most patches, providing confidence that the layer will contribute meaningful signal to the frozen geological prior.
</p>